# Industrial Machine Condition Classification
## Logistic Regression with Feature Engineering

**Target:** Predict `condition` (0 = normal, 1 = degraded, 2 = critical)  
**Approach:** EDA → feature engineering → logistic regression with polynomial interactions

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42

## 1. Load Data

In [2]:
train = pd.read_csv('data/student_dataset.csv')
test  = pd.read_csv('data/test.csv')
sub   = pd.read_csv('data/sample_submission.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

Train: (16000, 13)  |  Test: (4000, 12)


,unit_id,unit_type,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index,condition
0,UNT-184174,M,21.5,56.4,17.85,1454.0,58.2,9.099,30.0,29.31,57.19,1.768,0
1,UNT-236234,H,29.9,65.0,30.88,1720.0,54.6,10.101,13.0,23.93,36.94,1.081,0
2,UNT-247460,M,26.4,47.8,19.75,1552.0,38.4,6.328,89.0,13.52,38.22,1.031,0
3,UNT-176316,M,28.8,60.9,19.10,1513.0,49.1,7.768,59.0,24.62,46.17,1.011,0
4,UNT-105110,S,29.4,64.0,11.77,1536.0,52.6,8.405,36.0,27.83,94.72,1.851,1


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   unit_id          16000 non-null  object 
 1   unit_type        15978 non-null  object 
 2   ambient_temp     15953 non-null  float64
 3   process_temp     15955 non-null  float64
 4   coolant_flow     15939 non-null  float64
 5   shaft_speed      15968 non-null  float64
 6   drive_torque     15964 non-null  float64
 7   power_load       15975 non-null  float64
 8   component_wear   15944 non-null  float64
 9   thermal_stress   15950 non-null  float64
 10  mechanical_load  15955 non-null  float64
 11  vibration_index  15926 non-null  float64
 12  condition        16000 non-null  int64  
dtypes: float64(10), int64(1), object(2)
memory usage: 1.6+ MB


## 2. EDA

In [4]:
# --- Target distribution ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = train['condition'].value_counts().sort_index()
labels = {0: 'Normal', 1: 'Degraded', 2: 'Critical'}
sns.barplot(x=[labels[i] for i in counts.index], y=counts.values, ax=axes[0], palette='muted')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()), 
                     ha='center', va='bottom', fontsize=11)

unit_ct = train['unit_type'].value_counts()
sns.barplot(x=unit_ct.index, y=unit_ct.values, ax=axes[1], palette='pastel')
axes[1].set_title('Unit Type Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x7ef8d7982350> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 1100x400 with 2 Axes>

In [ ]:
# --- Feature distributions by condition ---
num_cols = ['vibration_index', 'component_wear', 'thermal_stress',
            'drive_torque', 'power_load', 'mechanical_load',
            'process_temp', 'shaft_speed']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
cond_labels = {0: 'Normal', 1: 'Degraded', 2: 'Critical'}
plot_df = train.copy()
plot_df['Condition'] = plot_df['condition'].map(cond_labels)

for ax, col in zip(axes, num_cols):
    sns.boxplot(data=plot_df, x='Condition', y=col, ax=ax,
                order=['Normal', 'Degraded', 'Critical'], palette='muted')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Feature Distributions by Condition', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))
corr = train[num_cols + ['condition']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# --- Missing values ---
missing = train.isnull().sum()
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing.values, ax=ax, palette='Reds_r')
ax.set_title('Missing Value Counts per Feature')
ax.set_ylabel('Missing count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()
print(f'Total missing: {train.isnull().sum().sum()} out of {train.size} cells ({train.isnull().sum().sum()/train.size*100:.2f}%)')

## 3. Feature Engineering

In [5]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Ordinal encode unit_type (S < M < H by capacity)
    df['unit_type_enc'] = df['unit_type'].map({'S': 0, 'M': 1, 'H': 2})

    # Temperature delta: how hard the process is working relative to ambient
    df['temp_delta'] = df['process_temp'] - df['ambient_temp']

    # Mechanical power proxy: torque × speed
    df['torque_x_speed'] = df['drive_torque'] * df['shaft_speed']

    # Compound wear signal
    df['wear_x_vibration'] = df['component_wear'] * df['vibration_index']

    # Thermal-mechanical stress product
    df['thermal_x_mech'] = df['thermal_stress'] * df['mechanical_load']

    # Coolant efficiency: how much heat is being managed per unit coolant
    df['heat_per_coolant'] = df['thermal_stress'] / (df['coolant_flow'] + 1e-6)

    # Wear per unit shaft speed
    df['wear_per_speed'] = df['component_wear'] / (df['shaft_speed'] + 1e-6)

    # Power per mechanical load
    df['power_per_load'] = df['power_load'] / (df['mechanical_load'] + 1e-6)

    # Drop non-numeric / ID columns
    df = df.drop(columns=['unit_id', 'unit_type'], errors='ignore')
    return df


train_fe = engineer_features(train.drop(columns=['condition']))
y        = train['condition'].values
test_fe  = engineer_features(test)

print('Feature matrix shape:', train_fe.shape)
train_fe.head()

Feature matrix shape: (16000, 18)


,ambient_temp,process_temp,coolant_flow,shaft_speed,drive_torque,power_load,component_wear,thermal_stress,mechanical_load,vibration_index,unit_type_enc,temp_delta,torque_x_speed,wear_x_vibration,thermal_x_mech,heat_per_coolant,wear_per_speed,power_per_load
0,21.5,56.4,17.85,1454.0,58.2,9.099,30.0,29.31,57.19,1.768,1.0,34.9,84622.8,53.040,1676.2389,1.642017,0.020633,0.159101
1,29.9,65.0,30.88,1720.0,54.6,10.101,13.0,23.93,36.94,1.081,2.0,35.1,93912.0,14.053,883.9742,0.774935,0.007558,0.273443
2,26.4,47.8,19.75,1552.0,38.4,6.328,89.0,13.52,38.22,1.031,1.0,21.4,59596.8,91.759,516.7344,0.684557,0.057345,0.165568
3,28.8,60.9,19.10,1513.0,49.1,7.768,59.0,24.62,46.17,1.011,1.0,32.1,74288.3,59.649,1136.7054,1.289005,0.038995,0.168248
4,29.4,64.0,11.77,1536.0,52.6,8.405,36.0,27.83,94.72,1.851,0.0,34.6,80793.6,66.636,2636.0576,2.364486,0.023437,0.088735


## 4. Preprocessing & Model Pipeline

In [6]:
# Impute → PolynomialFeatures (degree-2, interactions only) → Scale → Logistic Regression
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly',    PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('scaler',  StandardScaler()),
    ('clf',     LogisticRegression(max_iter=2000, C=0.5, solver='saga',
                                   random_state=SEED, n_jobs=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(pipeline, train_fe.values, y, cv=cv, scoring='accuracy', n_jobs=-1)

print(f'CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {np.round(cv_scores, 4)}')

CV Accuracy: 0.8809 ± 0.0024
Fold scores: [0.8784 0.8844 0.8784 0.8831 0.8803]


## 5. Train on Full Data & Evaluate

In [7]:
pipeline.fit(train_fe.values, y)

train_pred = pipeline.predict(train_fe.values)
print('=== Training Set Report ===')
print(classification_report(y, train_pred, target_names=['Normal', 'Degraded', 'Critical']))

=== Training Set Report ===
              precision    recall  f1-score   support

      Normal       0.89      0.94      0.91      8000
    Degraded       0.88      0.83      0.86      4000
    Critical       0.88      0.83      0.86      4000

    accuracy                           0.89     16000
   macro avg       0.89      0.87      0.88     16000
weighted avg       0.89      0.89      0.88     16000



In [8]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y, train_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Degraded', 'Critical'],
            yticklabels=['Normal', 'Degraded', 'Critical'])
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
ax.set_title('Confusion Matrix (Train)')
plt.tight_layout()
plt.show()

RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x7ef8d7982350> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 600x500 with 2 Axes>

## 6. Feature Importance (Logistic Regression Coefficients)

In [ ]:
# Get names after polynomial expansion
poly_step   = pipeline.named_steps['poly']
imputer     = pipeline.named_steps['imputer']
feat_names  = poly_step.get_feature_names_out(train_fe.columns.tolist())

# Sum absolute coefficients across classes → overall importance
coef_abs = np.abs(pipeline.named_steps['clf'].coef_).sum(axis=0)
top_n = 20
top_idx = np.argsort(coef_abs)[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=coef_abs[top_idx], y=feat_names[top_idx], ax=ax, palette='viridis')
ax.set_title(f'Top {top_n} Features by |Coefficient| Sum Across Classes')
ax.set_xlabel('Sum of |Coefficients|')
plt.tight_layout()
plt.show()

## 7. Generate Submission

In [ ]:
test_preds = pipeline.predict(test_fe.values)

submission = pd.DataFrame({
    'unit_id':   test['unit_id'],
    'condition': test_preds
})

submission.to_csv('data/submission_logistic_regression.csv', index=False)
print('Submission saved.')
print(f'Predicted distribution:\n{pd.Series(test_preds).value_counts().sort_index()}')
submission.head()